##  **Problem 1: Project description**

This question on this problem set is to come up with a project idea and write a paragraph
describing it. This doesn’t have to be completely worked out but I want to make sure that
have put in some time thinking about a possible topic and what that might involve (I won’t
hold you to this topic but if you do significantly change the project topic, please check with
me first). Please do include a bit of background but focus mostly on what you want to do.
There is a list of possible projects on the courseworks web page, but that should just be a
starting point – think big and feel free to email me or talk with me if you have questions.


### Answer:

## **Problem 2: Method Stability**


Consider the linear advection equation
$$
\frac{\partial a}{\partial t} + v \frac{\partial a}{\partial x} = 0
\tag{1}
$$

In class, we saw that an explicit first-order finite-difference discretization of this resulted in an unstable method. Now let us consider an implicit discretization of this same upwind method. To do this, we evaluate the spatial derivative at the new time rather than the old time so that the finite difference representation is
$$
\frac{a^{n+1}_j - a^n_j}{\Delta t}
= v\,\frac{a^{n+1}_{j+1} - a^{n+1}_{j-1}}{2\,\Delta x}
\tag{2}
$$

1. Perform linear stability analysis (as we did in class) and show that this method is stable for any choice of Courant number
   $$
   C \equiv \frac{v\,\Delta t}{\Delta x}.
   $$


2. Describe how you would solve this implicit discretization numerically with periodic boundary conditions. Write out the difference equations for a grid with $N=6$ in matrix form. You should find that the matrix is almost tridiagonal, except for a single element in a corner resulting from the periodicity.


3. You do not need to actually solve this system, but do some research into SciPy’s linear algebra framework and describe how you would solve it in principle (including which functions you would call).


## **Problem 3: Lax-Wendroff implementation**

The second-order Lax–Wendroff method is derived by Taylor expanding in time to O(Δt²) and replacing time derivatives using the PDE. Using centered, second‑order spatial differences yields a scheme that is second order in space and time. For linear advection, with subscripts as spatial indices and superscripts as time indices:
$$
a_i^{n+1}
= a_i^{n}
-\frac{C}{2}\left(a_{i+1}^{n}-a_{i-1}^{n}\right)
+\frac{C^{2}}{2}\left(a_{i+1}^{n}-2a_i^{n}+a_{i-1}^{n}\right).
$$
Here the first correction term corresponds to the unstable FTCS advection, while the second acts like positive diffusion. The Courant number is $C = v,\Delta t / \Delta x$. The method is stable when the Courant condition $C < 1$ holds; the added second‑derivative term counteracts the FTCS instability.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

1.  Implement this method on a cell-centered finite-difference grid (again following the
routine discussed in class if you wish). You can write your own code or adapt a toolkit
you find online (such as M. Zingale’s pyro framework), but be sure to give appropriate
credit (my own sense is that it may be easier to write your own code rather than learn
one of those frameworks but everyone’s experience is different)

In [ ]:

class FDGrid(object):
    def __init__(self, nx, ng, xmin=0.0, xmax=1.0):
        self.xmin = xmin
        self.xmax = xmax
        self.ng = ng
        self.nx = nx

        # python is zero-based; ilo/ihi mark the real data region
        self.ilo = ng
        self.ihi = ng + nx - 1

        # physical coordinates
        self.dx = (xmax - xmin) / (nx - 1)
        self.x = xmin + (np.arange(nx + 2 * ng) - ng) * self.dx

        # storage for the solution
        self.a = np.zeros((nx + 2 * ng), dtype=np.float64)

    def scratch_array(self):
        """return a scratch array dimensioned for our grid"""
        return np.zeros((self.nx + 2 * self.ng), dtype=np.float64)

    def fill_BCs(self):
        """fill a single ghost cell with periodic boundary conditions"""
        self.a[self.ilo - 1] = self.a[self.ihi - 1]
        self.a[self.ihi + 1] = self.a[self.ilo + 1]



2. Set up the following test problem. For the initial conditions, choose a Gaussian:
a(x, t = 0) = e−(x−0.5)2/0.12

In [ ]:

# create the grid
nx = [65,100]
ng = 1
g = FDGrid(nx, ng)

# define the CFL and speed
C = []
u = 1.0

# time info
dt = C * g.dx / u
t = 0.0
tmax = 1.0 * (g.xmax - g.xmin) / u

# initialize the data -- tophat
g.a[np.logical_and(g.x >= 0.333, g.x <= 0.666)] = 1.0
ainit = g.a.copy()

# evolution loop
anew = g.scratch_array()
while t < tmax:
    # fill the boundary conditions
    g.fill_BCs()

    # loop over zones (periodic); both endpoints lie on domain boundary
    for i in range(g.ilo, g.ihi + 1):
        # FTCS
        anew[i] = g.a[i] - 0.5 * C * (g.a[i + 1] - g.a[i - 1])
        # upwind (alternative)
        # anew[i] = g.a[i] - C * (g.a[i] - g.a[i - 1])

    # store the updated solution
    g.a[:] = anew[:]
    t += dt

plt.plot(g.x[g.ilo:g.ihi + 1], ainit[g.ilo:g.ihi + 1], ls=":")
plt.plot(g.x[g.ilo:g.ihi + 1], g.a[g.ilo:g.ihi + 1])
plt.savefig("fdadvect.png")
